# Photo Try-On - Stage 0b: dressing test on Kaggle's free GPU (run 2)

Puts a shirt and trousers on four photos of one person (front, two sides, back) with the open-source
**Leffa** try-on model. Run 1 showed it works and fits in memory; this run tests three fixes:

1. **Keep the head.** Leffa's mask for tops protects the face but not hair, and a beard counts as hair,
   so side views had the beard repainted. Here the head (face, hair, beard) is kept out of the repainted
   area, and everything outside that area is pasted back from the original photo.
2. **The back view.** Run 1 drew the wrong shirt on the back. Here the back uses Leffa's second model
   (trained on DressCode) for the top.
3. **Speed.** Both GPUs work at once, and the front view is repeated with fewer steps to see what that costs.

**Before running** (right-hand panel, *Session options*): Accelerator = **GPU T4 x2**, Internet = **On**.

In [ ]:
import time, os, glob, sys, shutil, threading, urllib.request, torch
T0 = time.time()
def stamp(what): print(f"[{time.time() - T0:6.0f}s] {what}", flush=True)

gpus = [f"{torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.0f} GB)" for i in range(torch.cuda.device_count())]
print("GPUs:", gpus or "NONE", "| torch", torch.__version__, "| python", sys.version.split()[0])
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15); print("Internet: on")
except Exception as e:
    print("Internet: OFF -", e)
assert gpus, "No GPU in this session. On Kaggle: verify your phone number (Settings), then set Accelerator to GPU T4 x2."

## 1. Get the model's code and the few packages Kaggle does not already have

In [ ]:
%cd /tmp
if not os.path.exists("Leffa"):
    !git clone -q --depth 1 https://github.com/franciszzj/Leffa
%cd /tmp/Leffa
!pip install -q fvcore omegaconf onnxruntime pycocotools av config cloudpickle einops peft 2>&1 | tail -3
stamp("code and packages ready")

## 2. Download the trained model files (about 16 GB; the 21 GB pose-transfer model is skipped)

In [ ]:
from huggingface_hub import snapshot_download
CK = "/tmp/ckpts"
snapshot_download(repo_id="franciszzj/Leffa", local_dir=CK,
                  ignore_patterns=["pose_transfer.pth", "stable-diffusion-xl-1.0-inpainting-0.1/*", "examples/*", "assets/*"])
stamp("model files downloaded")

## 3. Load the models: both try-on models on each GPU, so two photos can be dressed at once

In [ ]:
import numpy as np, cv2
from PIL import Image, ImageFilter
from leffa.transform import LeffaTransform
from leffa.model import LeffaModel
from leffa.inference import LeffaInference
from leffa_utils.densepose_predictor import DensePosePredictor
from leffa_utils.utils import resize_and_center, get_agnostic_mask_hd, get_agnostic_mask_dc
from preprocess.humanparsing.run_parsing import Parsing
from preprocess.openpose.run_openpose import OpenPose

densepose_predictor = DensePosePredictor(config_path=f"{CK}/densepose/densepose_rcnn_R_50_FPN_s1x.yaml",
                                         weights_path=f"{CK}/densepose/model_final_162be9.pkl")
parsing = Parsing(atr_path=f"{CK}/humanparsing/parsing_atr.onnx", lip_path=f"{CK}/humanparsing/parsing_lip.onnx")
openpose = OpenPose(body_model_path=f"{CK}/openpose/body_pose_model.pth")

N_GPU = torch.cuda.device_count()
WEIGHTS = {"viton_hd": "virtual_tryon.pth", "dress_code": "virtual_tryon_dc.pth"}
inference = {}                      # (gpu, model name) -> LeffaInference
for gpu in range(N_GPU):
    for name, weights in WEIGHTS.items():
        model = LeffaModel(pretrained_model_name_or_path=f"{CK}/stable-diffusion-inpainting",
                           pretrained_model=f"{CK}/{weights}", dtype="float16")
        inf = LeffaInference(model=model)          # puts it on the first GPU
        if gpu:                                    # then move it, and tell every part where it now lives
            inf.device = f"cuda:{gpu}"; inf.model.to(inf.device); inf.pipe.device = inf.device
        inference[(gpu, name)] = inf
        stamp(f"loaded {name} on GPU {gpu}")

def gpu_gb(): return [round(torch.cuda.max_memory_allocated(i) / 1e9, 1) for i in range(N_GPU)]
print("GPU memory in use after loading (GB):", gpu_gb())

## 4. The dressing function

Leffa's own steps - find the body parts and the pose, blank out the area the new garment covers, repaint
it from the garment picture - with two changes: the head is taken out of the repainted area, and
afterwards everything outside that area is put back from the original photo.

In [ ]:
HEAD = [1, 2, 3, 11]                 # hat, hair (a beard counts as hair), sunglasses, face
prep_lock = threading.Lock()         # the body-part and pose models are shared; one photo at a time

def dress(person, garment, kind, model_type=None, steps=30, accel=False, gpu=0, seed=42):
    """person, garment: PIL images. kind: 'upper_body' or 'lower_body'.
    Returns (picture, repainted-area mask, body-part map, seconds)."""
    started = time.time()
    model_type = model_type or ("viton_hd" if kind == "upper_body" else "dress_code")
    src = resize_and_center(person, 768, 1024).convert("RGB")
    ref = resize_and_center(garment, 768, 1024).convert("RGB")
    src_array = np.array(src)
    with prep_lock:
        model_parse, _ = parsing(src.resize((384, 512)))
        keypoints = openpose(src.resize((384, 512)))
        if model_type == "viton_hd":
            mask = get_agnostic_mask_hd(model_parse, keypoints, kind)
            densepose = Image.fromarray(densepose_predictor.predict_seg(src_array)[:, :, ::-1])
        else:
            mask = get_agnostic_mask_dc(model_parse, keypoints, kind)
            seg = densepose_predictor.predict_iuv(src_array)[:, :, 0:1]
            densepose = Image.fromarray(np.concatenate([seg] * 3, axis=-1))

    # Keep the head: out of the repainted area, with a small margin.
    parse = np.array(model_parse.resize((384, 512), Image.NEAREST))
    head = cv2.dilate(np.isin(parse, HEAD).astype(np.uint8), np.ones((5, 5), np.uint8))
    area = (np.array(mask.resize((384, 512), Image.NEAREST).convert("L")) > 127) & (head == 0)
    mask = Image.fromarray((area * 255).astype(np.uint8)).resize((768, 1024), Image.NEAREST)

    data = LeffaTransform()({"src_image": [src], "ref_image": [ref], "mask": [mask], "densepose": [densepose]})
    out = inference[(gpu, model_type)](data, ref_acceleration=accel, num_inference_steps=steps,
                                       guidance_scale=2.5, seed=seed, repaint=False)
    generated = out["generated_image"][0].convert("RGB")

    # Put the original back everywhere outside the repainted area: a soft edge, but none on the head.
    soft = mask.filter(ImageFilter.MaxFilter(9)).filter(ImageFilter.GaussianBlur(4))
    head_big = Image.fromarray(head * 255).resize((768, 1024), Image.NEAREST)
    soft = Image.composite(Image.new("L", soft.size, 0), soft, head_big)
    result = Image.composite(generated, src, soft)
    return result, mask, model_parse, time.time() - started

# Colours for the body-part map, to see how each photo was read (a beard should come out as hair or face).
PALETTE = np.array([[0,0,0],[128,0,0],[255,0,0],[0,85,0],[170,0,51],[255,85,0],[0,0,85],[0,119,221],[85,85,0],
                    [0,85,85],[85,51,0],[52,86,128],[0,128,0],[0,0,255],[51,170,221],[0,255,255],[85,255,170],
                    [170,255,85],[255,255,0]], np.uint8)
def debug_picture(person, mask, model_parse):
    src = resize_and_center(person, 768, 1024).convert("RGB")
    parts = Image.fromarray(PALETTE[np.clip(np.array(model_parse.resize((384, 512), Image.NEAREST)), 0, 18)]).resize((768, 1024), Image.NEAREST)
    red = Image.blend(src, Image.new("RGB", src.size, (255, 0, 0)), 0.55)
    sheet = Image.new("RGB", (768 * 2, 1024))
    sheet.paste(Image.blend(src, parts, 0.6), (0, 0)); sheet.paste(Image.composite(red, src, mask.convert("L")), (768, 0))
    return sheet

## 5. Run it: navy polo first, then khaki chinos on top of that result

Two photos are dressed at the same time, one per GPU. The back photo gets the garments' back pictures and
Leffa's second model for the top; the sides have no pictures of their own, so they get the front ones.

In [ ]:
found = glob.glob("/kaggle/input/**/person_front.jpg", recursive=True)
assert found, "Add the dataset with person_front.jpg etc. under Input (right-hand panel)."
IN = os.path.dirname(found[0]); OUT = "/kaggle/working/results"; os.makedirs(OUT, exist_ok=True)
load = lambda name: Image.open(f"{IN}/{name}.jpg")

#        view      garment face  model for the top
PLAN = [("front",  "front",      "viton_hd"),
        ("side_a", "front",      "viton_hd"),
        ("back",   "back",       "dress_code"),
        ("side_b", "front",      "viton_hd")]
rows, timings, errors = {}, [], []

def work(gpu, jobs):
    try:
        for view, face, top_model in jobs:
            person = load(f"person_{view}")
            top, top_mask, parts, t1 = dress(person, load(f"garment_navy_polo_{face}"), "upper_body", top_model, gpu=gpu)
            stamp(f"GPU {gpu} {view}: polo ({top_model}) in {t1:.0f}s")
            both, low_mask, _, t2 = dress(top, load(f"garment_khaki_chinos_{face}"), "lower_body", gpu=gpu)
            stamp(f"GPU {gpu} {view}: chinos in {t2:.0f}s")
            top.save(f"{OUT}/{view}_1_polo.jpg", quality=93); both.save(f"{OUT}/{view}_2_polo_chinos.jpg", quality=93)
            debug_picture(person, top_mask, parts).save(f"{OUT}/{view}_debug.jpg", quality=85)
            rows[view] = (resize_and_center(person, 768, 1024), top, both); timings.extend([(view, "polo", t1), (view, "chinos", t2)])
    except Exception as e:
        import traceback; errors.append(traceback.format_exc()); print(errors[-1])

wall = time.time()
threads = [threading.Thread(target=work, args=(g, PLAN[g::N_GPU])) for g in range(N_GPU)]
for t in threads: t.start()
for t in threads: t.join()
wall = time.time() - wall
assert not errors, "a view failed - see the traceback above"
stamp(f"all four views dressed: {wall:.0f}s of waiting for {len(timings)} pictures")

## 6. Speed test: the front polo again with fewer steps, with and without Leffa's faster mode

In [ ]:
speed = []
def trial(gpu, label, **kw):
    try:
        pic, _, _, t = dress(load("person_front"), load("garment_navy_polo_front"), "upper_body", gpu=gpu, **kw)
        pic.save(f"{OUT}/speed_{label}.jpg", quality=93); speed.append((label, t)); stamp(f"speed {label}: {t:.0f}s")
    except Exception as e:
        import traceback; print(traceback.format_exc())
trials = [("20_steps", dict(steps=20)), ("20_steps_fast_mode", dict(steps=20, accel=True)), ("12_steps", dict(steps=12)), ("12_steps_fast_mode", dict(steps=12, accel=True))]
def run_trials(gpu, mine):
    for label, kw in mine: trial(gpu, label, **kw)
threads = [threading.Thread(target=run_trials, args=(g, trials[g::N_GPU])) for g in range(N_GPU)]
for t in threads: t.start()
for t in threads: t.join()

## 7. Results

In [ ]:
import matplotlib.pyplot as plt
order = [p[0] for p in PLAN]
fig, axes = plt.subplots(len(order), 3, figsize=(12, 5.2 * len(order)))
for r, view in enumerate(order):
    person, top, both = rows[view]
    for c, (img, title) in enumerate(((person, f"{view}: photo"), (top, "navy polo"), (both, "polo + khaki chinos"))):
        axes[r, c].imshow(img); axes[r, c].set_title(title); axes[r, c].axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/sheet.jpg", dpi=70, bbox_inches="tight"); plt.show()

print("\nSeconds per picture (30 steps, two at a time):")
for view, what, t in sorted(timings): print(f"  {view:7s} {what:7s} {t:5.0f}s")
print(f"Waiting time for the whole outfit on four views: {wall:.0f}s")
print("Speed trials (front polo):", ", ".join(f"{label} {t:.0f}s" for label, t in sorted(speed)))
print(f"Whole notebook: {time.time() - T0:.0f}s   |   peak GPU memory (GB) per GPU: {gpu_gb()}")